In [ ]:
!git clone https://github.com/yao8839836/kg-bert.git
%cd kg-bert

!pip install transformers==2.5.1
!pip install -r requirements.txt

fatal: destination path 'kg-bert' already exists and is not an empty directory.
/content/drive/My Drive/colab/MEmbER/Northern_Ireland/kg-bert
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 622.0 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.6/64.6 kB 2.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.4/499.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 8.4 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.


In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import BertTokenizer, BertModel
from tqdm import tqdm
import torch.nn.functional as F
from math import log

In [ ]:
%cd kg-bert

/content/drive/MyDrive/colab/MEmbER/Northern_Ireland/kg-bert


In [ ]:
df = pd.read_csv('./data/northern_ireland.csv')

In [ ]:
df["population"] = df["population"].fillna(0)
df = df.replace("nan", np.nan)
df = df.replace("NaN", np.nan)
df = df.replace("None", np.nan)
df["placeType"] = df["placeType"].fillna("")
df["broaderPlaceType"] = df["broaderPlaceType"].fillna("")

In [ ]:
df.head(10)

,place,placeName,abstract,country,subject,broaderSubject,placeType,broaderPlaceType,population
0,http://dbpedia.org/resource/Bellaghy,Bellaghy,"Bellaghy (from Irish Baile Eachaidh, meaning '...",http://dbpedia.org/resource/Northern_Ireland,http://dbpedia.org/resource/Category:Villages_...,http://dbpedia.org/resource/Category:Villages_...,,,1063.0
1,http://dbpedia.org/resource/Bellaghy,Bellaghy,"Bellaghy (from Irish Baile Eachaidh, meaning '...",http://dbpedia.org/resource/Northern_Ireland,http://dbpedia.org/resource/Category:Villages_...,http://dbpedia.org/resource/Category:Geography...,,,1063.0
2,http://dbpedia.org/resource/Bellaghy,Bellaghy,"Bellaghy (from Irish Baile Eachaidh, meaning '...",http://dbpedia.org/resource/Northern_Ireland,http://dbpedia.org/resource/Category:Villages_...,http://dbpedia.org/resource/Category:Towns_and...,,,1063.0
3,http://dbpedia.org/resource/Bellaghy,Bellaghy,"Bellaghy (from Irish Baile Eachaidh, meaning '...",http://dbpedia.org/resource/Northern_Ireland,http://dbpedia.org/resource/Category:Mid-Ulste...,http://dbpedia.org/resource/Category:Districts...,,,1063.0
4,http://dbpedia.org/resource/Hillhall,Hillhall,Hillhall is a townland and non-nucleated villa...,http://dbpedia.org/resource/Northern_Ireland,http://dbpedia.org/resource/Category:Civil_par...,http://dbpedia.org/resource/Category:Civil_par...,,,118.0
5,http://dbpedia.org/resource/Hillhall,Hillhall,Hillhall is a townland and non-nucleated villa...,http://dbpedia.org/resource/Northern_Ireland,http://dbpedia.org/resource/Category:Civil_par...,http://dbpedia.org/resource/Category:Civil_par...,,,118.0
6,http://dbpedia.org/resource/Hillhall,Hillhall,Hillhall is a townland and non-nucleated villa...,http://dbpedia.org/resource/Northern_Ireland,http://dbpedia.org/resource/Category:Civil_par...,http://dbpedia.org/resource/Category:Barony_of...,,,118.0
7,http://dbpedia.org/resource/Hillhall,Hillhall,Hillhall is a townland and non-nucleated villa...,http://dbpedia.org/resource/Northern_Ireland,http://dbpedia.org/resource/Category:Civil_par...,http://dbpedia.org/resource/Category:Barony_of...,,,118.0
8,http://dbpedia.org/resource/Hillhall,Hillhall,Hillhall is a townland and non-nucleated villa...,http://dbpedia.org/resource/Northern_Ireland,http://dbpedia.org/resource/Category:Villages_...,http://dbpedia.org/resource/Category:Villages_...,,,118.0
9,http://dbpedia.org/resource/Hillhall,Hillhall,Hillhall is a townland and non-nucleated villa...,http://dbpedia.org/resource/Northern_Ireland,http://dbpedia.org/resource/Category:Villages_...,http://dbpedia.org/resource/Category:Geography...,,,118.0


In [ ]:
with open("data/places/entities.txt","w", encoding="utf-8") as f:
  for entity_id in df['place']:
    f.write(f"{entity_id}\n")

In [ ]:
# def uri_remove(uri):
#   return str(uri).strip().replace("http://dbpedia.org/resource/", "").replace("http://dbpedia.org/ontology/","").replace("Category:","").replace("_", " ")

# entity2text, entity2textlong = {}, {}
# for idx, row in df.iterrows():
#   entity_id = row['place']
#   if entity_id not in entity2text:
#     text = f"{row['placeName']}. "
#     longtext = f"{row['abstract']} "
#     longtext += f"Population: {row['population']}"
#     entity2text[entity_id] = text
#     entity2textlong[entity_id] = longtext


# for c in df["country"].unique():
#     if c not in entity2text:
#       entity2text[c] = f"{c}"
#     if c not in entity2textlong:
#       entity2textlong[c] = f"{c}. A country."


# for s in df["subject"].unique():
#     if s not in entity2text:
#         entity2text[s] = f"{s}"
#     if s not in entity2textlong:
#         entity2textlong[s] = f"{s}. A subject category."

# # 3. Add subjects
# for s in df["broaderSubject"].unique():
#     if s not in entity2text:
#         entity2text[s] = f"{s}"
#     if s not in entity2textlong:
#         entity2textlong[s] = f"{s}. A parent subject category."

# # 4. Add city types
# for t in df["placeType"].unique():
#     if t not in entity2text:
#         entity2text[t] = f"{t}"
#     if t not in entity2textlong:
#         entity2textlong[t] = f"{t}. A type of place."

# # 5. Add broader place types
# for bt in df["broaderPlaceType"].unique():
#     if bt not in entity2text:
#         entity2text[bt] = f"{bt}"
#     if bt not in entity2textlong:
#         entity2textlong[bt] = f"{bt}. A type of place."


# with open("data/places/entity2text.txt","w", encoding="utf-8") as f:
#   for entity, text in entity2text.items():
#     f.write(f"{entity}\t{text}\n")

# with open("data/places/entity2textlong.txt","w", encoding="utf-8") as f:
#   for entity, textlong in entity2textlong.items():
#     f.write(f"{entity}\t{textlong}\n")

In [ ]:
import unicodedata

def normalize_uri(uri):
    import unicodedata

    uri = str(uri).strip()

    uri = uri.encode().decode("unicode_escape")
    uri = unicodedata.normalize("NFKC", uri)
    uri = uri.replace("–", "-").replace("—", "-")

    return uri


def uri_remove(uri):
    uri = normalize_uri(uri)

    return (
        uri.replace("http://dbpedia.org/resource/", "")
           .replace("http://dbpedia.org/ontology/", "")
           .replace("Category:", "")
           .replace("_", " ")
    )


entity2text, entity2textlong = {}, {}

for idx, row in df.iterrows():

    entity_id = normalize_uri(row['place'])

    if entity_id not in entity2text:

        text = f"{row['placeName']}. "

        longtext = f"{row['abstract']} "
        longtext += f"Population: {row['population']}"

        entity2text[entity_id] = text
        entity2textlong[entity_id] = longtext



for c in df["country"].dropna().unique():

    c = normalize_uri(c)

    if c not in entity2text:
        entity2text[c] = uri_remove(c)

    if c not in entity2textlong:
        entity2textlong[c] = f"{uri_remove(c)}. A country."



for s in df["subject"].dropna().unique():

    s = normalize_uri(s)

    if s not in entity2text:
        entity2text[s] = uri_remove(s)

    if s not in entity2textlong:
        entity2textlong[s] = f"{uri_remove(s)}. A subject category."



for s in df["broaderSubject"].dropna().unique():

    s = normalize_uri(s)

    if s not in entity2text:
        entity2text[s] = uri_remove(s)

    if s not in entity2textlong:
        entity2textlong[s] = f"{uri_remove(s)}. A parent subject category."



for t in df["placeType"].dropna().unique():

    t = normalize_uri(t)

    if t not in entity2text:
        entity2text[t] = uri_remove(t)

    if t not in entity2textlong:
        entity2textlong[t] = f"{uri_remove(t)}. A type of place."



for bt in df["broaderPlaceType"].dropna().unique():

    bt = normalize_uri(bt)

    if bt not in entity2text:
        entity2text[bt] = uri_remove(bt)

    if bt not in entity2textlong:
        entity2textlong[bt] = f"{uri_remove(bt)}. A broader type of place."


with open("data/places/entity2text.txt", "w", encoding="utf-8") as f:
    for entity, text in entity2text.items():
        f.write(f"{entity}\t{text}\n")


with open("data/places/entity2textlong.txt", "w", encoding="utf-8") as f:
    for entity, textlong in entity2textlong.items():
        f.write(f"{entity}\t{textlong}\n")

In [ ]:
df = df.map(normalize_uri)

In [ ]:
triples = pd.read_csv("data/northern_ireland.tsv", sep="\t", names=['head','relation','tail'])

for col in ["head", "relation", "tail"]:

    triples[col] = (
        triples[col]
        .astype(str)
        .str.strip()
        .str.replace(r"^<|>$", "", regex=True)
    )

unstructured_relations = {'http://dbpedia.org/ontology/abstract', 'http://dbpedia.org/ontology/populationTotal', 'http://www.w3.org/2000/01/rdf-schema#label'}
triples = triples[~triples['relation'].isin(unstructured_relations)]

In [ ]:
relations_dict = {"http://dbpedia.org/ontology/abstract":"abstract",
                  "http://dbpedia.org/ontology/country":"country",
                  "http://www.w3.org/2000/01/rdf-schema#label":"the name of the place",
                  "http://dbpedia.org/ontology/populationTotal":"population",
                  "http://dbpedia.org/ontology/type":"the type of the place",
                  "http://www.w3.org/1999/02/22-rdf-syntax-ns#type":"an instance of a class",
                  "http://purl.org/dc/terms/subject":"category of the place",
                  "http://www.w3.org/2004/02/skos/core#broader":"parent category"
                  }

In [ ]:
with open("data/places/relation2text.txt", "w", encoding="utf-8") as f:
    for r, desc in relations_dict.items():
        f.write(f"{r}\t{desc}\n")

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(triples, test_size=0.2, random_state=42)
valid_df, test_df = train_test_split(test_df, test_size=0.5, random_state=42)

In [ ]:
train_df.to_csv("data/places/train.tsv",sep="\t", index=False, header=False)
test_df.to_csv("data/places/test.tsv",sep="\t", index=False, header=False)
valid_df.to_csv("data/places/dev.tsv",sep="\t", index=False, header=False)

In [ ]:
all_triple_entities = set()
for split_file in ["data/places/train.tsv","data/places/test.tsv","data/places/dev.tsv"]:
  with open(split_file) as f:
    for line in f:
      parts = line.strip().split("\t")
      if len(parts) >= 3:
        all_triple_entities.add(parts[0])
        all_triple_entities.add(parts[2])

missing = all_triple_entities - set(entity2text.keys())
print(f"Entities missing from entity2text: {len(missing)}")
for ent in missing:
  label = ent.split("/")[-1].replace("_", " ").replace("Category:", "")
  entity2text[ent] = label
  entity2textlong[ent] = label

with open(os.path.join("data/places/", "entity2text.txt"), "w", encoding="utf-8") as f:
  for entity, text in entity2text.items():
    f.write(f"{entity}\t{text}\n")

with open(os.path.join("data/places/", "entity2textlong.txt"), "w", encoding="utf-8") as f:
  for entity, text in entity2textlong.items():
    f.write(f"{entity}\t{text}\n")

print(f"entity2text entries after fix: {len(entity2text)}")

Entities missing from entity2text: 16
entity2text entries after fix: 1431


In [ ]:
!python run_bert_triple_classifier.py --task_name kg --do_train  --do_eval --do_predict --data_dir ./data/places --bert_model bert-base-uncased --max_seq_length 20 --train_batch_size 32 \
--learning_rate 5e-5 --num_train_epochs 3.0 --output_dir ./places-data/  --gradient_accumulation_steps 1 --eval_batch_size 512

In [ ]:
# df = pd.read_csv("data/Irish_populated_places.csv")

In [ ]:
df.head(1)

,place,placeName,abstract,country,subject,broaderSubject,placeType,broaderPlaceType,population
0,http://dbpedia.org/resource/Bellaghy,Bellaghy,"Bellaghy (from Irish Baile Eachaidh, meaning '...",http://dbpedia.org/resource/Northern_Ireland,http://dbpedia.org/resource/Category:Villages_...,http://dbpedia.org/resource/Category:Villages_...,,,1063.0


In [ ]:
df['subject'] = df['subject'].str.replace(
    'http://dbpedia.org/resource/Category:', '', regex=False
).map(lambda x: x.replace('_',' '))

In [ ]:
df['broaderSubject'] = df['broaderSubject'].fillna('').str.replace(
    'http://dbpedia.org/resource/Category:', '', regex=False
).str.replace('_', ' ')

In [ ]:
df['country'] = df['country'].fillna('').str.replace(
    'http://dbpedia.org/resource/', '', regex=False
).str.replace('_', ' ')

In [ ]:
df['placeType'] = df['placeType'].fillna('').str.replace(
    'http://dbpedia.org/resource/', '', regex=False
).str.replace('_', ' ')

In [ ]:
df['broaderPlaceType'] = df['broaderPlaceType'].fillna('').str.replace(
    'http://dbpedia.org/ontology/', '', regex=False
)

In [ ]:
agg_df = df.groupby("place", as_index=False).agg({
    "placeName": "first",
    "abstract": "first",
    "country": "first",
    "subject": lambda x: "; ".join(set(x.dropna().astype(str))),
    "broaderSubject": lambda x: "; ".join(set(x.dropna().astype(str))),
    "placeType": lambda x: "; ".join(set(x.dropna().astype(str))),
    "broaderPlaceType": lambda x: "; ".join(set(x.dropna().astype(str))),
    "population": "first"
})

In [ ]:
agg_df.head(10)

,place,placeName,abstract,country,subject,broaderSubject,placeType,broaderPlaceType,population
0,http://dbpedia.org/resource/Acravally,Acravally,"Acravally is a townland in County Antrim, Nort...",Northern Ireland,Barony of Cary; Townlands of County Antrim,Townlands of Northern Ireland by county; Geogr...,,,0
1,"http://dbpedia.org/resource/Acton,_County_Armagh","Acton, County Armagh",Acton is a small village and townland of 22 ac...,Northern Ireland,Townlands of County Armagh; Civil parish of Ba...,Geography of County Armagh; Towns and villages...,,,0
2,"http://dbpedia.org/resource/Aghaboy,_County_An...","Aghaboy, County Antrim","Aghaboy is a townland in County Antrim, Northe...",Northern Ireland,Civil Parish of Drummaul; Townlands of County ...,Townlands of Northern Ireland by county; Geogr...,,,0
3,http://dbpedia.org/resource/Aghacommon,Aghacommon,"Aghacommon (from Irish Achadh CamÃÂ¡n, meanin...",Northern Ireland,Townlands of County Armagh; Villages in County...,Towns and villages in Northern Ireland by coun...,,,0
4,http://dbpedia.org/resource/Aghacullion,Aghacullion,Aghacullion (from Irish Achadh an Chuilinn 'f...,Northern Ireland,Townlands of County Down,Townlands of Northern Ireland by county; Geogr...,,,0
5,http://dbpedia.org/resource/Aghadowey,Aghadowey,Aghadowey (from Irish Achadh Dubhthaigh 'Duff...,Northern Ireland,Causeway Coast and Glens district; Villages in...,Towns and villages in Northern Ireland by coun...,,,0
6,http://dbpedia.org/resource/Aghadrumsee,Aghadrumsee,Aghadrumsee (from Irish Achadh Dhruim Saileac...,Northern Ireland,Townlands of County Fermanagh; Fermanagh and O...,Geography of County Fermanagh; Towns and villa...,,,0
7,http://dbpedia.org/resource/Aghagallon,Aghagallon,"Aghagallon (from Irish Achadh Gallan , meaning...",Northern Ireland,Villages in County Antrim; Civil parishes of C...,Towns and villages in Northern Ireland by coun...,,,824.0
8,http://dbpedia.org/resource/Aghaginduff,Aghaginduff,"Aghaginduff (from Irish Achadh Cinn Duibh, mea...",Northern Ireland,Townlands of County Tyrone; Civil parish of Ki...,Civil parishes of County Tyrone; Townlands of ...,,,0
9,http://dbpedia.org/resource/Aghakeeran,Aghakeeran,Aghakeeran (from Irish Achadh Chaorthainn 'fi...,Northern Ireland,Townlands of County Fermanagh; Fermanagh and O...,Townlands of Northern Ireland by county; Civil...,,,0


In [ ]:
def entity_to_text(row):
    text = f"{row['placeName']}. {row['abstract']}. Country: {row['country']}. "
    text += f"category of the place: {row['subject']}. "
    text += f"parent category: {row['broaderSubject']}. "
    text += f"Place types: {row['placeType']}. "
    text += f"Broader place types: {row['broaderPlaceType']}. "
    text += f"Population: {row['population']}"
    return text

agg_df['entity_text'] = agg_df.apply(entity_to_text, axis=1)

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("places-data/")
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


entity_repr = {}

for idx, row in tqdm(agg_df.iterrows(), total=len(agg_df), desc="Encoding entities"):
    text = row['entity_text']
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze()
        entity_repr[row['place']] = cls_embedding.cpu()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: places-data/
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Encoding entities: 100%|██████████| 617/617 [07:55<00:00,  1.30it/s]


In [ ]:
def compute_similar_places(entity_repr, top_k=10):
    entities = list(entity_repr.keys())
    all_embeddings = torch.stack([entity_repr[e] for e in entities])  # [N, D]
    sims = F.cosine_similarity(all_embeddings.unsqueeze(1), all_embeddings.unsqueeze(0), dim=-1)

    similar_places_dict = {}
    for i, e in enumerate(entities):
        sim_scores = sims[i]
        topk = torch.topk(sim_scores, k=top_k+1)  # +1 to skip itself
        indices = topk.indices.tolist()
        similar_entities = [entities[j] for j in indices if j != i][:top_k]
        similar_places_dict[e] = similar_entities
    return similar_places_dict

In [ ]:
similar_places = compute_similar_places(entity_repr, top_k=10)

In [ ]:
ground_truth = pd.read_csv("./data/northern_ireland_ground_truth.csv")
ground_truth_filtered = (
    ground_truth
    .sort_values(
        by=["geographical_entity_1", "f_score"],
        ascending=[True, False]
    )
    .groupby("geographical_entity_1")
    .head(20)
)
y_true_dict = ground_truth_filtered.groupby('geographical_entity_1')['geographical_entity_2'].apply(list).to_dict()

In [ ]:
y_true_list = [
    (e1, e2, score)
    for e1, e2, score in ground_truth_filtered[['geographical_entity_1', 'geographical_entity_2', 'f_score']].values
]

In [ ]:
def evluation(k, y_true_dict, similar_places_dict):
  invalid_places = []
  # Compute metrics
  precisions, recalls, ndcgs, hits, map_scores,mrrs = [], [], [], [], [], []
  place_idxs = list(y_true_dict.keys())

  for pid in place_idxs:
      if pid not in similar_places_dict or len(similar_places_dict[pid]) < k:
          invalid_places.append(pid)
          continue
      pred_list, rel_set = similar_places_dict[pid][:k], y_true_dict[pid]
      pred_real = "pid:"+str(pid)+' '+"pred_list:"+str(pred_list)+' '+"rel_set:"+str(rel_set)

      if len(pred_list) == 0:
          continue


      dcg = 0.0
      hit_num = 0.0
      for i in range(len(pred_list)):
          if pred_list[i] in rel_set:
              dcg += 1. / (log(i + 2) / log(2))
              hit_num += 1
      # idcg
      idcg = 0.0
      for i in range(min(len(rel_set), len(pred_list))):
          idcg += 1. / (log(i + 2) / log(2))
      ndcg = dcg / idcg
      recall = hit_num / len(rel_set)
      precision = hit_num / len(pred_list)
      hit = 1.0 if hit_num > 0.0 else 0.0

      #map
      map_score = 0.0
      num_hits = 0.0
      score = 0.0
      for i,p in enumerate(pred_list):
          if p in rel_set and p not in pred_list[:i]:
              num_hits+=1.0
              score+=num_hits/(i+1.0)
      map_score = score/min(len(rel_set),k)
      #map_score = score / min(len(rel_set), len(pred_list)) if len(rel_set) > 0 else 0.0

      #MRR
      rr = 0.0
      for i, p in enumerate(pred_list):
        if p in rel_set:
          rr = 1.0 / (i + 1.0)
          break

      ndcgs.append(ndcg)
      recalls.append(recall)
      precisions.append(precision)
      hits.append(hit)
      map_scores.append(map_score)
      mrrs.append(rr)

  avg_precision = np.mean(precisions) * 100
  avg_recall = np.mean(recalls) * 100
  avg_ndcg = np.mean(ndcgs) * 100
  avg_hit = np.mean(hits) * 100
  avg_map = np.mean(map_scores) * 100
  avg_mrr = np.mean(mrrs) * 100

  print("invalid places:", str(len(invalid_places)))
  print('MAP={:.3f} | NDCG={:.3f} |  Recall={:.3f} | Precision={:.3f} | Hits={:.3f} | MRR={:.3f}'.format(
          avg_map, avg_ndcg, avg_recall, avg_precision, avg_hit, avg_mrr))

In [ ]:
evluation(k=10, y_true_dict=y_true_dict, similar_places_dict=similar_places)